# 🔍 Anomaly Detection — Baseline Nhanh (Fixed Format)
**PatchCore + WideResNet-50** — Chạy ~3-5 phút trên T4
Tự động tính threshold & xuất chuẩn format `sample_id, category, label` (0/1)

## 1. Imports & Config

In [13]:
import os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════
# CHỈNH PATH Ở ĐÂY — Kaggle tự giải nén zip rồi
# ════════════════════════════════════════════════
TRAIN_DIR  = '/kaggle/input/datasets/nnt1810/ahihi12344/dataset_train/dataset_train'
TEST_DIR   = '/kaggle/input/datasets/nnt1810/ahihi12344/public_test/public_test'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
CORESET_RATIO = 0.1
K_NN = 3

random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'Device: {DEVICE}')

Device: cuda


## 2. Load data paths

In [14]:
# ── Train ──
train_img_dir = os.path.join(TRAIN_DIR, 'train')
train_paths = {}
for d in sorted(Path(train_img_dir).iterdir()):
    if d.is_dir() and d.name.startswith('category_'):
        imgs = sorted([str(p) for p in d.glob('*.jpg')])
        train_paths[d.name] = imgs
        print(f'{d.name}: {len(imgs)} ảnh')

# ── Test ──
test_df = pd.read_csv(os.path.join(TEST_DIR, 'test.csv'))
test_df['full_path'] = test_df['relative_path'].apply(
    lambda x: os.path.join(TEST_DIR, x)
)
print(f'\nTest: {len(test_df)} ảnh')

category_01: 664 ảnh
category_02: 664 ảnh
category_03: 302 ảnh
category_04: 660 ảnh
category_05: 660 ảnh
category_06: 210 ảnh

Test: 480 ảnh


## 3. Dataset & Backbone

In [15]:
class ImgDataset(Dataset):
    def __init__(self, paths, tf):
        self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB'))

tf = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1)
        self.layer2 = m.layer2
        self.layer3 = m.layer3
        for p in self.parameters(): p.requires_grad = False
        self.eval()

    @torch.no_grad()
    def forward(self, x):
        h = self.stem(x)
        f2 = self.layer2(h)
        f3 = self.layer3(f2)
        f3 = F.interpolate(f3, size=f2.shape[2:], mode='bilinear', align_corners=False)
        out = torch.cat([f2, f3], dim=1)
        B,C,H,W = out.shape
        return out.permute(0,2,3,1).reshape(B, H*W, C)

backbone = Backbone().to(DEVICE)
print('✓ WideResNet-50 loaded')

✓ WideResNet-50 loaded


## 4. 🚀 Train & Predict
Tính điểm bất thường (Anomaly Score) thô cho từng ảnh.

In [16]:
%%time
import os
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
def extract_feats(paths):
    dl = DataLoader(ImgDataset(paths, tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2, pin_memory=True)
    feats = []
    for batch in dl:
        f = backbone(batch.to(DEVICE))
        B,N,D = f.shape
        feats.append(f.reshape(B*N, D).cpu())
    return torch.cat(feats)

# ĐÃ SỬA HÀM NÀY ĐỂ CHẠY TRONG 0.1 GIÂY
def coreset(feats, ratio):
    n = feats.shape[0]
    k = max(int(n * ratio), 1)
    if k >= n: return feats
    indices = torch.randperm(n)[:k]
    return feats[indices]

raw_scores = {}  
checkpoints = {} 

for cat in sorted(train_paths.keys()):
    cat_test = test_df[test_df['category'] == cat]
    if len(cat_test) == 0: continue

    print(f'\n── {cat} ──')

    # 1. TRÍCH XUẤT ĐẶC TRƯNG & LƯU CHECKPOINT
    print('  [Train] Extracting features...')
    feats = extract_feats(train_paths[cat])
    bank = coreset(feats, CORESET_RATIO)
    del feats
    
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{cat}_memory_bank.pt')
    torch.save(bank, ckpt_path)
    print(f'  [CheckPoint] Saved Memory Bank {bank.shape[0]:,} patches -> {ckpt_path}')

    # 2. TÍNH ANOMALY SCORE
    print('  [Predict] Calculating anomaly scores...')
    bank_gpu = bank.to(DEVICE).float()
    dl = DataLoader(ImgDataset(cat_test['full_path'].tolist(), tf),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    cat_scores = []
    for batch in dl:
        f = backbone(batch.to(DEVICE))
        for i in range(f.shape[0]):
            d = torch.cdist(f[i].float(), bank_gpu)
            topk, _ = torch.topk(d, K_NN, dim=1, largest=False)
            cat_scores.append(topk.mean(dim=1).max().item())

    for sid, sc in zip(cat_test['sample_id'].values, cat_scores):
        raw_scores[sid] = sc

    del bank, bank_gpu; torch.cuda.empty_cache()
    arr = np.array(cat_scores)
    print(f'  [Stats] Score phân bố: min={arr.min():.3f} | max={arr.max():.3f} | mean={arr.mean():.3f}')


── category_01 ──
  [Train] Extracting features...
  [CheckPoint] Saved Memory Bank 52,057 patches -> /kaggle/working/checkpoints/category_01_memory_bank.pt
  [Predict] Calculating anomaly scores...
  [Stats] Score phân bố: min=26.711 | max=51.749 | mean=34.676

── category_02 ──
  [Train] Extracting features...
  [CheckPoint] Saved Memory Bank 52,057 patches -> /kaggle/working/checkpoints/category_02_memory_bank.pt
  [Predict] Calculating anomaly scores...
  [Stats] Score phân bố: min=24.944 | max=46.116 | mean=31.198

── category_03 ──
  [Train] Extracting features...
  [CheckPoint] Saved Memory Bank 23,676 patches -> /kaggle/working/checkpoints/category_03_memory_bank.pt
  [Predict] Calculating anomaly scores...
  [Stats] Score phân bố: min=37.649 | max=49.364 | mean=43.594

── category_04 ──
  [Train] Extracting features...
  [CheckPoint] Saved Memory Bank 51,744 patches -> /kaggle/working/checkpoints/category_04_memory_bank.pt
  [Predict] Calculating anomaly scores...
  [Stats] S

## 5. Thresholding & Định dạng CSV (QUAN TRỌNG)
Chuyển anomaly_score thành nhãn 0/1 (0=normal, 1=anomaly).
Ở đây dùng chiến thuật: Nếu score > (mean + 0.5*std) của category đó thì là anomaly (1).

In [17]:
# Gán raw score vào dataframe
test_df['anomaly_score'] = test_df['sample_id'].map(raw_scores)

final_labels = []
for cat in test_df['category'].unique():
    cat_mask = test_df['category'] == cat
    scores = test_df.loc[cat_mask, 'anomaly_score'].values
    
    # Tính threshold tự động cho category này
    # Baseline đơn giản: threshold = mean + 0.5 * std
    threshold = np.mean(scores) + 0.5 * np.std(scores)
    
    # Áp dụng threshold: >= threshold là 1 (anomaly), < threshold là 0 (normal)
    labels = (scores >= threshold).astype(int)
    test_df.loc[cat_mask, 'label'] = labels
    
    print(f"{cat}: threshold={threshold:.3f} -> phát hiện {sum(labels)} anomalies / {len(labels)} ảnh")

# Chuyển label sang số nguyên
test_df['label'] = test_df['label'].astype(int)

# Chỉ lấy đúng 3 cột theo yêu cầu của đề: sample_id, category, label
submission = test_df[['sample_id', 'category', 'label']].copy()

out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)

print(f'\n✓ Saved {out_path} ({len(submission)} rows)')
print(submission.head())

category_03: threshold=44.904 -> phát hiện 27 anomalies / 80 ảnh
category_02: threshold=33.661 -> phát hiện 19 anomalies / 80 ảnh
category_06: threshold=34.322 -> phát hiện 17 anomalies / 80 ảnh
category_01: threshold=37.268 -> phát hiện 21 anomalies / 80 ảnh
category_05: threshold=41.405 -> phát hiện 23 anomalies / 80 ảnh
category_04: threshold=34.664 -> phát hiện 29 anomalies / 80 ảnh

✓ Saved /kaggle/working/submission.csv (480 rows)
                  sample_id     category  label
0  img_0091c3f65178ce922a3e  category_03      0
1  img_03189bfdbe7f6f7d9196  category_02      0
2  img_03581d373757407a0da0  category_06      0
3  img_036730c826566bf0124f  category_02      0
4  img_03daea60de93eb864e08  category_01      0
